# Notebook 00 — Data Exploration

This notebook:
1. Loads the IAM Handwriting Database (words split)
2. Loads the FUNSD dataset from HuggingFace
3. Visualises 50 random IAM word samples
4. Logs: avg dimensions, character set size, samples per split, word length distribution
5. Runs baseline TrOCR on 100 IAM test samples (no fine-tuning) → baseline CER/WER

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import jiwer

print('Imports OK')

In [ ]:
# ── Load IAM Dataset ─────────────────────────────────────────────────────────
# Requires manual registration + download from:
# https://www.fki.inf.unibe.ch/databases/iam-handwriting-database
# Place files under ../data/iam/

IAM_DIR = Path('../data/iam')
assert IAM_DIR.exists(), f'IAM data not found at {IAM_DIR}. Download it first.'

# Read words.txt annotation file
words_file = IAM_DIR / 'ascii' / 'words.txt'
samples = []
with open(words_file) as f:
    for line in f:
        if line.startswith('#'): continue
        parts = line.strip().split(' ')
        if len(parts) < 9: continue
        word_id = parts[0]
        label = parts[-1]
        # Construct image path: {IAM_DIR}/words/a01/a01-000u/a01-000u-00-00.png
        parts_id = word_id.split('-')
        img_path = IAM_DIR / 'words' / parts_id[0] / f'{parts_id[0]}-{parts_id[1]}' / f'{word_id}.png'
        if img_path.exists():
            samples.append({'id': word_id, 'label': label, 'path': str(img_path)})

print(f'Total IAM word samples found: {len(samples)}')

In [ ]:
# ── Dataset Statistics ────────────────────────────────────────────────────────
import random

labels = [s['label'] for s in samples]
word_lengths = [len(l) for l in labels]
char_set = set(''.join(labels))

print(f'Character set size:        {len(char_set)}')
print(f'Unique labels:             {len(set(labels))}')
print(f'Mean word length:          {np.mean(word_lengths):.2f} chars')
print(f'Max word length:           {max(word_lengths)} chars')
print(f'Character set:             {sorted(char_set)}')

In [ ]:
# ── Visualise 50 Random Samples ───────────────────────────────────────────────
subset = random.sample(samples, min(50, len(samples)))
fig, axes = plt.subplots(5, 10, figsize=(20, 8))
for ax, s in zip(axes.flat, subset):
    img = Image.open(s['path']).convert('RGB')
    ax.imshow(img)
    ax.set_title(s['label'], fontsize=7)
    ax.axis('off')
plt.suptitle('50 Random IAM Word Samples', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/iam_samples.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Baseline TrOCR (no fine-tuning) on 100 test samples ──────────────────────
from src.ocr import TrOCREngine

engine = TrOCREngine()  # loads microsoft/trocr-base-handwritten

test_samples = random.sample(samples, 100)
predictions, references = [], []

for s in test_samples:
    img = Image.open(s['path'])
    candidates = engine.recognise(img)
    pred = candidates[0].word if candidates else ''
    predictions.append(pred)
    references.append(s['label'])

baseline_cer = jiwer.cer(references, predictions)
baseline_wer = jiwer.wer(references, predictions)

print(f'Baseline CER (no fine-tuning): {baseline_cer:.4f} ({baseline_cer*100:.1f}%)')
print(f'Baseline WER (no fine-tuning): {baseline_wer:.4f} ({baseline_wer*100:.1f}%)')
print('These are your reference numbers. All improvements are measured against these.')